In [ ]:
# 00_preprocessing — Onset strength function experiments
# Goal: find a better ODF that improves onset F1 on the full 127-file training set.
# Each experiment swaps only the ODF; peak picking and evaluation stay identical.
# Best method here → integrate into src/features.py and re-run 01_pipeline.ipynb.
import sys
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import librosa
from scipy.ndimage import median_filter, maximum_filter1d, gaussian_filter1d
import mir_eval
from tqdm import tqdm

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

from src.config import config
from src.data_loader import DataLoader

config.paths.base_dir = project_root
config.paths.data_raw_dir = project_root / 'data' / 'raw'
config.paths.data_processed_dir = project_root / 'data' / 'processed'
config.paths.submissions_dir = project_root / 'submissions'

SR = config.audio.sample_rate   # 22050
HOP = config.audio.onset_hop_length  # 512

print(f'Project root: {project_root}')
print(f'SR={SR}, hop={HOP}')

In [ ]:
# Load training data and pre-cache audio (avoids repeated disk reads)
loader = DataLoader()
train_dir = config.paths.data_processed_dir / 'train'
train_data = loader.load_train(train_dir)
print(f'Training files: {len(train_data)}')

print('Pre-loading audio...')
audio_cache: dict = {}
for stem, info in tqdm(train_data.items()):
    y, sr = loader.load_audio(info['wav'])
    if y is not None:
        audio_cache[stem] = (y, sr)
print(f'Cached {len(audio_cache)} files')

In [ ]:
# Shared LFSF three-condition peak picker — identical to OnsetDetector.detect().
# Takes a pre-computed ODF instead of raw audio so we can swap ODFs freely.

def lfsf_pick(odf: np.ndarray, sr: int, hop_length: int,
              threshold: float, smoothing: float = 1.0) -> np.ndarray:
    if smoothing > 0:
        odf = gaussian_filter1d(odf, sigma=smoothing)
    fps = sr / hop_length
    N = len(odf)
    w_max = max(1, int(round(0.030 * fps)))
    w_avg = max(1, int(round(0.100 * fps)))
    wait  = max(1, int(round(0.050 * fps)))
    peaks, last = [], -wait - 1
    for n in range(N):
        x = odf[n]
        if x < odf[max(0, n - w_max):min(N, n + w_max + 1)].max():
            continue
        if x < odf[max(0, n - w_avg):min(N, n + w_avg + 1)].mean() + threshold:
            continue
        if n - last <= wait:
            continue
        peaks.append(n)
        last = n
    return librosa.frames_to_time(
        np.array(peaks, dtype=int), sr=sr, hop_length=hop_length
    )


def eval_onset_f1(odf_fn, hop_length: int = HOP,
                  threshold: float = 0.08) -> float:
    """
    Evaluate an ODF function on all cached training files with onset annotations.
    odf_fn: callable(y, sr) -> 1-D numpy array
    Returns mean onset F1 over annotated files.
    """
    f1s = []
    for stem, (y, sr) in audio_cache.items():
        if not train_data[stem].get('onsets'):
            continue
        odf = odf_fn(y, sr)
        pred = lfsf_pick(odf, sr, hop_length, threshold)
        f, _, _ = mir_eval.onset.f_measure(
            np.array(train_data[stem]['onsets']), pred, window=0.05
        )
        f1s.append(f)
    return float(np.mean(f1s)) if f1s else 0.0


def sweep_thresholds(odf_fn, thresholds, hop_length: int = HOP, label: str = ''):
    """Sweep threshold values and print onset F1 for each."""
    best_f1, best_thresh = 0.0, thresholds[0]
    for thresh in thresholds:
        f1 = eval_onset_f1(odf_fn, hop_length=hop_length, threshold=thresh)
        marker = ' <--' if f1 > best_f1 else ''
        print(f'  threshold={thresh:.4f}: F1={f1:.4f}{marker}')
        if f1 > best_f1:
            best_f1, best_thresh = f1, thresh
    print(f'  [{label}] best: F1={best_f1:.4f} at threshold={best_thresh}')
    return best_f1, best_thresh


THRESHOLDS = [0.08, 0.04, 0.02, 0.01, 0.005, 0.002]
print('Helpers ready.')

In [ ]:
# ── BASELINE: current superflux (no maximum filter, γ = 1) ─────────────────
# Mirrors src/features.py:FeatureExtractor.superflux() exactly.

def odf_baseline(y, sr,
                 hop_length=HOP, n_mels=64, fmin=30.0, fmax=17000.0):
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=2048, hop_length=hop_length,
        n_mels=n_mels, fmin=fmin, fmax=fmax
    )
    log_mel = np.log1p(mel)                          # γ = 1
    diff = np.diff(log_mel, axis=1, prepend=log_mel[:, :1])
    odf = np.sum(np.maximum(diff, 0), axis=0)
    if odf.max() > 0:
        odf /= odf.max()
    return odf

print('BASELINE (current superflux, γ=1, no max-filter):')
baseline_best_f1, baseline_best_thresh = sweep_thresholds(
    odf_baseline, THRESHOLDS, label='baseline'
)

In [ ]:
# ── EXPERIMENT A: Real SuperFlux with maximum filter (Böck et al. 2012) ────
#
# The key missing piece in the current implementation:
#   1. Log compression with γ = 100 instead of γ = 1
#      → spreads the dynamic range so quiet onsets are visible
#   2. Maximum filter of size μ along the FREQUENCY axis before differencing
#      → suppresses vibrato: vibrato shifts energy between adjacent frequency
#        bins at the SAME time instant. The max filter means a bin must exceed
#        the maximum of its neighbors from the previous frame — a mere shift
#        between bins produces zero positive diff, not a spurious detection.
#      → filter is along axis=0 (n_mels) NOT axis=1 (time)

def odf_superflux_real(y, sr,
                       hop_length=HOP, n_mels=82,
                       fmin=27.5, fmax=16744.0,
                       gamma=100.0, mu=3):
    """
    True SuperFlux (Böck et al. ICASSP 2012).
    gamma: log compression strength (default 100)
    mu:    max-filter size in frequency bins (default 3)
    """
    mel = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=2048, hop_length=hop_length,
        n_mels=n_mels, fmin=fmin, fmax=fmax
    )
    log_mel = np.log1p(gamma * mel)          # (n_mels, n_frames)
    # Maximum filter along FREQUENCY axis (axis=0)
    max_filt = maximum_filter1d(log_mel, size=mu, axis=0)
    # Temporal difference: current frame vs max-filtered previous frame
    diff = log_mel[:, 1:] - max_filt[:, :-1]
    diff = np.pad(diff, ((0, 0), (1, 0)), mode='constant')
    odf = np.sum(np.maximum(diff, 0), axis=0)
    if odf.max() > 0:
        odf /= odf.max()
    return odf

print('EXPERIMENT A: Real SuperFlux (γ=100, μ=3):')
expA_best_f1, expA_best_thresh = sweep_thresholds(
    odf_superflux_real, THRESHOLDS, label='real superflux'
)

In [ ]:
# ── EXPERIMENT A2: Sweep γ and μ to find best parameters ───────────────────

print('Sweeping γ (threshold=0.01, μ=3):')
for gamma in [1, 10, 50, 100, 200]:
    fn = lambda y, sr, g=gamma: odf_superflux_real(y, sr, gamma=g, mu=3)
    f1 = eval_onset_f1(fn, threshold=0.01)
    print(f'  gamma={gamma:>4}: F1={f1:.4f}')

print('\nSweeping μ (threshold=0.01, γ=100):')
for mu in [1, 2, 3, 5, 7, 11]:
    fn = lambda y, sr, m=mu: odf_superflux_real(y, sr, gamma=100, mu=m)
    f1 = eval_onset_f1(fn, threshold=0.01)
    print(f'  mu={mu:>2}: F1={f1:.4f}')

In [ ]:
# ── EXPERIMENT B: Percussive isolation via median-filter HPSS ──────────────
#
# Idea: onsets are driven by the PERCUSSIVE content (drums, plucks, attacks).
# Harmonic content (sustained tones, vibrato) creates false positives.
# HPSS separates them by median-filtering the spectrogram:
#   - Harmonic component: long median along TIME (smooth over rapid changes)
#   - Percussive component: long median along FREQUENCY (smooth over tone content)
# We compute onset flux only on the percussive soft-masked spectrogram.

def odf_percussive(y, sr,
                   hop_length=HOP,
                   harmonic_win=17, percussive_win=17, gamma=100.0):
    """
    HPSS-style: soft Wiener mask to isolate percussive content.
    harmonic_win:    median filter size along TIME axis (frames)
    percussive_win:  median filter size along FREQUENCY axis (bins)
    """
    S = np.abs(librosa.stft(y, n_fft=2048, hop_length=hop_length))
    H = median_filter(S, size=(1, harmonic_win))    # smooth time → harmonic
    P = median_filter(S, size=(percussive_win, 1))  # smooth freq → percussive
    # Soft Wiener mask
    mask_p = P ** 2 / (H ** 2 + P ** 2 + 1e-10)
    S_perc = np.log1p(gamma * S * mask_p)
    diff = np.diff(S_perc, axis=1, prepend=S_perc[:, :1])
    odf = np.sum(np.maximum(diff, 0), axis=0)
    if odf.max() > 0:
        odf /= odf.max()
    return odf

print('EXPERIMENT B: Percussive isolation (HPSS-style):')
expB_best_f1, expB_best_thresh = sweep_thresholds(
    odf_percussive, THRESHOLDS, label='percussive'
)

In [ ]:
# ── EXPERIMENT C: Pre-emphasis ──────────────────────────────────────────────
#
# Pre-emphasis (y[n] - α·y[n-1]) boosts high frequencies before spectrogram
# computation.  Onsets are transient → strong high-frequency content.
# This can sharpen the ODF peak at attack times.

def apply_preemphasis(y: np.ndarray, coef: float = 0.97) -> np.ndarray:
    return np.concatenate([[y[0]], y[1:] - coef * y[:-1]])

def odf_preemphasis(y, sr, coef=0.97, **kwargs):
    return odf_superflux_real(apply_preemphasis(y, coef), sr, **kwargs)

print('EXPERIMENT C: Pre-emphasis (α=0.97) + Real SuperFlux:')
expC_best_f1, expC_best_thresh = sweep_thresholds(
    odf_preemphasis, THRESHOLDS, label='preemphasis'
)

print('\nSweeping α (threshold=0.01):')
for coef in [0.5, 0.7, 0.9, 0.95, 0.97, 0.99]:
    fn = lambda y, sr, c=coef: odf_preemphasis(y, sr, coef=c)
    f1 = eval_onset_f1(fn, threshold=0.01)
    print(f'  alpha={coef}: F1={f1:.4f}')

In [ ]:
# ── EXPERIMENT D: Combination — pre-emphasis + HPSS mask + Real SuperFlux ──

def odf_combined(y, sr, coef=0.97, gamma=100.0, mu=3,
                 harmonic_win=17, percussive_win=17,
                 hop_length=HOP):
    y_pre = apply_preemphasis(y, coef)
    S = np.abs(librosa.stft(y_pre, n_fft=2048, hop_length=hop_length))
    H = median_filter(S, size=(1, harmonic_win))
    P = median_filter(S, size=(percussive_win, 1))
    mask_p = P ** 2 / (H ** 2 + P ** 2 + 1e-10)
    S_perc = S * mask_p
    mel_filters = librosa.filters.mel(
        sr=sr, n_fft=2048, n_mels=82, fmin=27.5, fmax=16744.0
    )
    mel_perc = mel_filters @ S_perc          # (n_mels, n_frames)
    log_mel = np.log1p(gamma * mel_perc)
    # Maximum filter along FREQUENCY axis (axis=0)
    max_filt = maximum_filter1d(log_mel, size=mu, axis=0)
    diff = log_mel[:, 1:] - max_filt[:, :-1]
    diff = np.pad(diff, ((0, 0), (1, 0)), mode='constant')
    odf = np.sum(np.maximum(diff, 0), axis=0)
    if odf.max() > 0:
        odf /= odf.max()
    return odf

print('EXPERIMENT D: Pre-emphasis + HPSS mask + Real SuperFlux:')
expD_best_f1, expD_best_thresh = sweep_thresholds(
    odf_combined, THRESHOLDS, label='combined'
)

In [ ]:
# ── EXPERIMENT E: Shorter hop length ───────────────────────────────────────
#
# hop=512 → ~23 ms/frame.  The eval window is 50 ms.
# hop=256 → ~12 ms/frame — finer time resolution, may shift detections
# closer to ground truth.

print('EXPERIMENT E: hop_length comparison (Real SuperFlux, threshold=0.01):')
for hl in [256, 512, 1024]:
    fn = lambda y, sr, h=hl: odf_superflux_real(y, sr, hop_length=h)
    f1 = eval_onset_f1(fn, hop_length=hl, threshold=0.01)
    fps = SR / hl
    print(f'  hop_length={hl}: F1={f1:.4f}  ({fps:.1f} fps)')

In [ ]:
# ── VISUALISE: compare ODFs on a single file ────────────────────────────────

stem = list(audio_cache.keys())[0]
y, sr = audio_cache[stem]
gt_onsets = np.array(train_data[stem]['onsets'])

odf_a = odf_baseline(y, sr)
odf_b = odf_superflux_real(y, sr)
odf_c = odf_percussive(y, sr)

t = librosa.frames_to_time(np.arange(len(odf_a)), sr=sr, hop_length=HOP)
t_lim = 8  # seconds
mask = t <= t_lim

fig, axes = plt.subplots(3, 1, figsize=(14, 7), sharex=True)
labels = ['Baseline (γ=1, no max-filter)', 'Real SuperFlux (γ=100, μ=3)', 'Percussive HPSS']
odfs   = [odf_a, odf_b, odf_c]

for ax, odf, label in zip(axes, odfs, labels):
    ax.plot(t[mask], odf[mask], lw=0.8, color='steelblue')
    for o in gt_onsets[gt_onsets <= t_lim]:
        ax.axvline(o, color='red', alpha=0.6, lw=0.8, ls='--')
    ax.set_ylabel('ODF')
    ax.set_title(label)

axes[-1].set_xlabel('Time (s)')
plt.suptitle(f'{stem} — red dashes = ground truth onsets')
plt.tight_layout()
plt.show()

In [ ]:
# ── SUMMARY TABLE ───────────────────────────────────────────────────────────

print('=' * 60)
print(f'{"Method":<40} {"Best F1":>8}  {"Threshold":>10}')
print('-' * 60)

rows = [
    ('Baseline (current, γ=1)',          baseline_best_f1, baseline_best_thresh),
    ('Exp A — Real SuperFlux (γ=100)',   expA_best_f1,    expA_best_thresh),
    ('Exp B — Percussive HPSS',          expB_best_f1,    expB_best_thresh),
    ('Exp C — Pre-emphasis + SuperFlux', expC_best_f1,    expC_best_thresh),
    ('Exp D — Combined',                 expD_best_f1,    expD_best_thresh),
]

best_overall = max(rows, key=lambda r: r[1])
for name, f1, thresh in rows:
    marker = ' <<<' if f1 == best_overall[1] else ''
    print(f'{name:<40} {f1:>8.4f}  {thresh:>10.4f}{marker}')

print('=' * 60)
print(f'\nWinner: {best_overall[0]}')
print(f'  Onset F1 = {best_overall[1]:.4f} (threshold={best_overall[2]})')
print(f'  Baseline = {baseline_best_f1:.4f}')
print(f'  Delta    = {best_overall[1] - baseline_best_f1:+.4f}')
print('\nNext step: integrate winner into src/features.py and evaluate')
print('full pipeline (onset + beat + tempo) in 01_pipeline.ipynb.')